# Bond cash flows and pricing

Research question: Do dated cash flows, accrued interest and YTM reconcile?

Inputs: frozen Treasury 2024–2025 CSVs. Rates are decimals, prices per 100 face. Release 1.0.0. No live signal.

In [1]:
import sys,json
from pathlib import Path
from datetime import date
import numpy as np
R=Path.cwd()
if R.name=='notebooks':R=R.parent
sys.path.insert(0,str(R/'src'));sys.path.insert(0,str(R/'scripts'))
from curve_lab.data import load_history
from curve_lab.bonds import Bond
from curve_lab.curves import bootstrap,par_rate,TENORS,KEYS,fit_nelson_siegel
from curve_lab.risk import bond_risk,hedge,carry_bridge
from curve_lab.forecast import predict,evaluate
from review_display import table,figure
if 'display_bundle' not in globals():
    from IPython.display import display
    def display_bundle(bundle):display(bundle,raw=True)
D=json.loads((R/'artifacts/release/research.json').read_text());rows,receipts=load_history(R)
print('Run',D['run_id'],'| source through',D['as_of'],'| hypothetical bond valuations')


Run p4-662f58a64e8e | source through 2025-12-31 | hypothetical bond valuations


## Generate the complete dated schedule

In [2]:
b=Bond(date(2030,12,31),.04);s=date(2025,12,31);c=bootstrap(rows[-1]['par']);cf=b.cashflows(s);display_bundle(table(['Payment','ACT/365 time','Yield periods','Cashflow'],[[x['date'],x['time'],x['periods'],x['amount']] for x in cf]));assert len(cf)==10

Payment,ACT/365 time,Yield periods,Cashflow
2026-06-30,0.49589041,1,2
2026-12-31,1,2,2
2027-06-30,1.4958904,3,2
2027-12-31,2,4,2
2028-06-30,2.4986301,5,2
2028-12-31,3.0027397,6,2
2029-06-30,3.4986301,7,2
2029-12-31,4.0027397,8,2
2030-06-30,4.4986301,9,2
2030-12-31,5.0027397,10,102


## Solve the price-yield round trip

In [3]:
p=b.curve_price(s,c);y=b.ytm(s,p);ai=b.accrued(s);display_bundle(table(['Item','Value'],[['Dirty',p],['Accrued',ai],['Clean',p-ai],['YTM',y],['Round-trip difference',b.yield_price(s,y)-p]]));assert abs(b.yield_price(s,y)-p)<1e-9

Item,Value
Dirty,101.21553
Accrued,0
Clean,101.21553
YTM,0.037312572
Round-trip difference,-1.1368684e-13


## Challenge a mid-period date and coupon date

In [4]:
inside=date(2026,2,28);a=b.accrued(inside);display_bundle(table(['Case','Accrued'],[['28 Feb 2026',a],['30 Jun 2026',b.accrued(date(2026,6,30))]]));assert abs(a-2*59/181)<1e-12;assert b.accrued(date(2026,6,30))==0;assert abs(Bond(date(2030,12,31),0).yield_price(s,.05)-100/1.025**10)<1e-10

Case,Accrued
28 Feb 2026,0.6519337
30 Jun 2026,0


## Interpretation

Clean plus accrued equals dirty. Actual/Actual coupon periods differ deliberately from Actual/365 curve time. The schedule is regular and unadjusted, with no actual-security or stub-convention claim.

Canonical evidence: `artifacts/release/research.json`; source receipts: `data/manifests/sources.json`.